This project builds an experience‑based rating model for motor third‑party liability insurance using the freMTPL dataset. The goal is to estimate expected claim frequency and severity for different customer segments, and combine them into a pure premium model. This allows us to understand how risk varies by rating factor (age, vehicle power, region, bonus‑malus, etc.) and derive indicated premiums for each segment.

In [6]:
#------------------------------------
#1. Load the data
#------------------------------------

import pandas as pd

freq_data = pd.read_csv('../data/raw/freMTPLfreq.csv') 
sev_data = pd.read_csv('../data/raw/freMTPLsev.csv')

#We merge the frequency and severity data on the 'PolicyID' column to create a combined dataset for analysis.
data = pd.merge(freq_data, sev_data, on=['PolicyID'], how='left')


In [7]:

#------------------------------------
#2. Explore the data
#------------------------------------
data.info()
print(data.head())
data.describe()
#check for missing values
data.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 413960 entries, 0 to 413959
Data columns (total 11 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   PolicyID     413960 non-null  int64  
 1   ClaimNb      413960 non-null  int64  
 2   Exposure     413960 non-null  float64
 3   Power        413960 non-null  object 
 4   CarAge       413960 non-null  int64  
 5   DriverAge    413960 non-null  int64  
 6   Brand        413960 non-null  object 
 7   Gas          413960 non-null  object 
 8   Region       413960 non-null  object 
 9   Density      413960 non-null  int64  
 10  ClaimAmount  16181 non-null   float64
dtypes: float64(2), int64(5), object(4)
memory usage: 34.7+ MB
   PolicyID  ClaimNb  Exposure Power  CarAge  DriverAge  \
0         1        0      0.09     g       0         46   
1         2        0      0.84     g       0         46   
2         3        0      0.52     f       2         38   
3         4        0      0.45 

PolicyID            0
ClaimNb             0
Exposure            0
Power               0
CarAge              0
DriverAge           0
Brand               0
Gas                 0
Region              0
Density             0
ClaimAmount    397779
dtype: int64

Here we check the data to see if there is anything missing and to see if the data looks good.

In [8]:
#------------------------------------
#3. Calculate actuarial metrics
#------------------------------------

#frequency = number of claims / number of policies
data['frequency'] = data['ClaimNb']/data['Exposure']
#severity (only for claims > 0)
data['severity'] = data['ClaimAmount']/data['ClaimNb']
#Pure premium
data['pure_premium'] = data['ClaimAmount']/data['Exposure']

data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 413960 entries, 0 to 413959
Data columns (total 14 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   PolicyID      413960 non-null  int64  
 1   ClaimNb       413960 non-null  int64  
 2   Exposure      413960 non-null  float64
 3   Power         413960 non-null  object 
 4   CarAge        413960 non-null  int64  
 5   DriverAge     413960 non-null  int64  
 6   Brand         413960 non-null  object 
 7   Gas           413960 non-null  object 
 8   Region        413960 non-null  object 
 9   Density       413960 non-null  int64  
 10  ClaimAmount   16181 non-null   float64
 11  frequency     413960 non-null  float64
 12  severity      16181 non-null   float64
 13  pure_premium  16181 non-null   float64
dtypes: float64(5), int64(5), object(4)
memory usage: 44.2+ MB


Claim frequency measures how often claims occur, adjusted for how long each policy was insured.
Frequency = Number of Claims/Exposure

Claim severity measures how expensive claims are, but only for policies that actually had a claim.
Severity = Total Claims Cost/Number of Claims

Pure premium represents the expected annual cost of claims for a policyholder. It combines both frequency and severity.

Pure Premium = Frequency * Severity

Or directly from the data:

Pure Premium = Total Claim Cost/Exposure




In [9]:
# Save the prepared data to a CSV file
data.to_csv('../data/processed/prepared_data.csv', index=False)